## TIFF plot

In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png")


In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse)
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)
# ---
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.file <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
        } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename){
    # nchar(filename)
    filename.length <- nchar(filename)
    
    # Get the extension ("." + 2 letters)
    # substr(filename,filename.length-2, filename.length)
    
    # Get the name without the extension ("." + 2 letters)
    start <- 1
    end <- filename.length-3
    unzip.filename <- substr(filename,start, end)

    
    if(!file.exists(unzip.filename)){
 
        # print(global_topo_tiff_gz)
        # print(global_topo_tiff)
        
        R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}

# ==============================================================================
# FILE INFO
# ==============================================================================
# print(file.info( global_topo_tiff_gz ))


# Ice concentration

In [ ]:
# 2.2 DOWNLOAD COPERNICUS NetCDF ----

# Sea ice area & Sea ice concentration estimates as retrieved by the algorithm,
# and that were edited away by the various filters
# system("python API_Copernicus.py")

# OR

# Use 'Copernicus marine data' output as an input
# ice_netcdf <- "./data/osisaff_obs-si_glo_phy_sic-south_my_amsr_cdr_P1D-m_1778859355280.nc"
ice_netcdf <- "./data/Copernicus marine data.netcdf"
# ice_netcdf <- move.file("Copernicus marine data.netcdf", dataDir)
nc_iceC_file <- nc_open(ice_netcdf)


In [ ]:
#  Extract the coordinates
dim_ice_lon <- ncvar_get(nc_iceC_file, "longitude") # nc_iceC_file$dim[[3]]$vals . longitude
dim_ice_lat <- ncvar_get(nc_iceC_file, "latitude") # nc_iceC_file$dim[[2]]$vals . latitude
dim_ice_time <- ncvar_get(nc_iceC_file, "time") # nc_iceC_file$dim[[1]]$vals . time
dim_ice <- ncvar_get(nc_iceC_file, "ice_conc")
dim_raw_ice <- ncvar_get(nc_iceC_file, "raw_ice_conc_values")



In [ ]:
t_units <- ncatt_get(nc_iceC_file, "time", "units")

#sea_water_potential_concentration
T <- c("ice_conc", "raw_ice_conc_values")

# convert time -- split the time units string into fields
t_ustr <- strsplit(t_units$value, " ")
t_dstr <- strsplit(unlist(t_ustr)[3], "-")
date <- ymd(t_dstr) + dseconds(dim_ice_time)

T_array_ice <- dim_ice
# Quick map plot

# set the time step
t <- 1 #temperature on 2003-01-01
T_slice <- T_array_ice[,,t]


In [ ]:
image(dim_ice_lon, dim_ice_lat, T_slice, col = rev(brewer.pal(10,"RdBu")))

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/collection/IceConcentration.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/collection/IceConcentration.pdf")
}

# Plot Ice Concentration
image(x = dim_ice_lon, y = dim_ice_lat, z = T_slice, col = rev(brewer.pal(10,"RdBu")))

# Save the plot ---
dev.off()

In [ ]:
# Create a set of lonxlat pairs of values, one for each element in the Temp_array
grid <- expand.grid(lon=dim_ice_lon, lat=dim_ice_lat)  

cutpts <- c(12,13,14,15,16,17,18,19,20)   # set colorbar
levelplot(T_slice ~ lon * lat,
          data=grid, region=TRUE,
          pretty=T, at=cutpts, cuts=9,
          col.regions=(rev(brewer.pal(9,"RdBu"))), contour=0,
          xlab = "Longitude", ylab = "Latitude",
          main = "Sea Water Potential Temperature (°C)"
          )

In [ ]:
ice_raw <- ncvar_get(nc_iceC_file, "ice_conc", raw_datavals = TRUE)

# Get attributes
fill_value <- ncatt_get(nc_iceC_file, "ice_conc", "_FillValue")$value
scale      <- ncatt_get(nc_iceC_file, "ice_conc", "scale_factor")$value

ice_raw[ice_raw == fill_value] <- NA
ice <- ice_raw * scale


In [ ]:
ice_coords <- expand.grid(dim_ice_lon, dim_ice_lat,date)
ice_matrix <- cbind(ice_coords, as.vector(dim_ice))
names(ice_matrix) <- c("lon", "lat", "time", "iceC")


In [ ]:
# Function to replace NaN with column median
df_clean <- lapply(ice_matrix, function(iceC) {
    if (is.numeric(iceC)) {
      # Compute median excluding NA and NaN
      med <- median(iceC, na.rm = TRUE)
      
      # Replace NaN values with median
      iceC[is.nan(iceC)] <- med
    }
    return(iceC)}
)

In [ ]:
nc_close(nc_iceC_file)

In [ ]:
table_filename <- paste(outputDir,"Ice_CopernicusMarineData.tsv", sep="/")
write.table(ice_matrix, table_filename, row.names=FALSE, sep="\t")

# Filter the Dataframe

In [ ]:
# ----------------
# Filter Out the outerbound values
Antarctic_iceC_coords <- as.data.frame(df_clean) %>%
  dplyr::mutate(
    month = format(time, "%m") # Apply month format
  ) %>%
  dplyr::filter(
    lon >= 30, lon <= 150, lat >= -70, lat <= -60,
    (month %in% c("01", "02", "03", "12")) | (month == "04" & format(time, "%d") <= "01")
  ) %>%
    dplyr::group_by(lon, lat) %>% # Groups ONLY BY COORDINATES
    dplyr::summarise(
    ice_mean = mean(iceC, na.rm = TRUE), # Compute the average Ice concentration in summer
    .groups = "drop"
  )


In [ ]:
table_filename <- paste(outputDir,"filter_Ice_CopernicusMarineData.tsv", sep="/")
write.table(Antarctic_iceC_coords, table_filename, row.names=FALSE, sep="\t")
